 

<!-- Main div with background color and padding -->

<div style="background-color:#e6f2ff; padding: 20px;">

<!-- Image with margin and size adjustments -->

<img src="https://images.pexels.com/photos/2908984/pexels-photo-2908984.jpeg?auto=compress&cs=tinysrgb&w=1260&h=750&dpr=1" alt="Image Alt Text" style="display:block; margin:auto; width:40%;"/>

<!-- Main title -->

<h1 style="font-family:Verdana; color:#333366; text-align:center; font-size:4em;">Learning Agency Lab - Automated Essay Scoring 2.0</h1>

<!-- Subtitle -->

<h3 style="font-family:Verdana; color:#666699; text-align:center; font-size:3em;">Improve upon essay scoring algorithms to improve student learning outcomes

</h3>

<!-- Author name -->

<h4 style="font-family:Verdana; text-align:center; font-size:2em;">by Jack Donohue</h4>

<!-- Section headers -->

<!-- Project Description -->

## <p style="font-family:JetBrains Mono; font-weight:bold; letter-spacing: 2px; color:#243139; font-size:140%; text-align:left;padding: 0px; border-bottom: 3px solid #000000"><a name="install-packages"></a>Project Description</p>

<p style="background-color:#e6f2ff;">Goal of the Competition & Link</p>

## <p style="font-family:JetBrains Mono; font-weight:bold; letter-spacing: 2px; color:#243139; font-size:140%; text-align:left;padding: 0px; border-bottom: 3px solid #000000"><a name="install-packages"></a>Goal of the Competition</p>

<p style="background-color:#e6f2ff;">Goal of the Competition & Link</p>


    
## <p style="font-family:JetBrains Mono; font-weight:bold; letter-spacing: 2px; color:#243139; font-size:140%; text-align:left;padding: 0px; border-bottom: 3px solid #000000"><a name="install-packages"></a>Table of Contents</p>

1. [Install Packages](#install-packages)
2. [Variable Settings](#var-settings)
3. [Kaggle Setup](#kaggle-setup)
4. [Data Loading](#data-loading)
5. [Exploratory Data Analysis (EDA)](#eda)
6. [Preprocessing](#preprocessing)
7. [Training](#training)
8. [Inference](#inference)
9. [Model Evaluation](#model-evaluation)

<br></br>

 

In [ ]:
 
# Import Packages
import shutup; shutup.please()
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras_tuner as kt
import seaborn as sns

from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk import pos_tag, ne_chunk
from textblob import TextBlob

from textstat import flesch_reading_ease, smog_index

import spacy
from collections import Counter
from gensim import corpora, models
import pyLDAvis.gensim as gen
import pyLDAvis
import re

# Machine Learning & Data Preprocessing

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Deep Learning

from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Gensim
# from gensim.models import Word2Vec, KeyedVectors

# Progress bar
from tqdm import tqdm

# Keras Tuner
from keras_tuner.tuners import RandomSearch

# # Setting logging levels and environment variables
# tf.get_logger().setLevel(logging.ERROR)
# os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

from textstat import flesch_reading_ease

# <p style="font-family:JetBrains Mono; font-weight:bold; letter-spacing: 2px; color:#243139; font-size:140%; text-align:left;padding: 0px; border-bottom: 3px solid #000000"> <a name="data-loading"></a>Data Loading</p>

### Competition data was cleaned in the 'clean-text-data.ipynb' notebook and loaded here for further processing and training

In [ ]:
# Data Loading
# Placeholder for data loading code
# train = pd.read_parquet('data/cleaned_train.parquet')

train = pd.read_parquet('result.parquet')

# test = pd.read_parquet('data/cleaned_test.parquet')

In [ ]:
train.head()

In [ ]:
# test.head()

# <p style="font-family:JetBrains Mono; font-weight:bold; letter-spacing: 2px; color:#243139; font-size:140%; text-align:left;padding: 0px; border-bottom: 3px solid #000000"> <a name="preprocessing"></a>Preprocessing</p>

In [ ]:
# helper functions to tokenize by word and by sentence and add to the dataframe
import nltk

def tokenize_by_word(text):
    return nltk.word_tokenize(text)

def tokenize_by_sentence(text):
    return nltk.sent_tokenize(text)

train['tokenized_word'] = train['corrected_text'].apply(tokenize_by_word)
train['tokenized_sentence'] = train['corrected_text'].apply(tokenize_by_sentence)



In [ ]:
# # helper function to lemmatize, remove stopwords and punctuation, and other important text preprocessing steps then add to the dataframe

# from nltk.corpus import stopwords
# from nltk.tokenize import word_tokenize, sent_tokenize
# from nltk.stem import WordNetLemmatizer
# import string

# def preprocess_text_by_word(text):
#     stop_words = set(stopwords.words('english'))
#     lemmatizer = WordNetLemmatizer()
#     text = text.lower()
#     text = re.sub(r'\d+', '', text)
#     text = text.translate(str.maketrans('', '', string.punctuation))
#     text = word_tokenize(text)
#     text = [lemmatizer.lemmatize(word) for word in text if word not in stop_words]
#     text = ' '.join(text)
#     return text

# def preprocess_text_by_sentence(text):
#     stop_words = set(stopwords.words('english'))
#     lemmatizer = WordNetLemmatizer()
#     text = text.lower()
#     text = re.sub(r'\d+', '', text)
#     text = text.translate(str.maketrans('', '', string.punctuation))
#     text = sent_tokenize(text)
#     text = [word_tokenize(sentence) for sentence in text]
#     text = [[lemmatizer.lemmatize(word) for word in sentence if word not in stop_words] for sentence in text]
#     text = [' '.join(sentence) for sentence in text]
#     return text

# train['tokenized_word'] = train['clean_text'].apply(preprocess_text_by_word)
# train['tokenized_sentence'] = train['clean_text'].apply(preprocess_text_by_sentence)

# train.head()

# <p style="font-family:JetBrains Mono; font-weight:bold; letter-spacing: 2px; color:#243139; font-size:140%; text-align:left;padding: 0px; border-bottom: 3px solid #000000"> <a name="eda"></a>Exploratory Data Analysis (EDA)</p>

In [ ]:
# Helper functions for EDA

def topic_modeling(texts, num_topics=5):
    """
    Perform topic modeling using LDA.
    """
    dictionary = corpora.Dictionary(texts)
    corpus = [dictionary.doc2bow(text) for text in tqdm(texts, desc='Creating corpus', leave=False)]
    lda_model = models.LdaModel(corpus, num_topics=num_topics, id2word=dictionary, passes=15)
    lda_display = gen.prepare(lda_model, corpus, dictionary)
    return lda_display

def sentiment_analysis(texts):
    """
    Perform sentiment analysis on the text using TextBlob.
    """
    return texts.apply(lambda x: TextBlob(x).sentiment.polarity)

def text_complexity(texts):
    """
    Calculate the Flesch Reading Ease score of the text using textstat.
    """
    return texts.apply(lambda x: flesch_reading_ease(x))


In [ ]:
tokenized_words = pd.Series(train['tokenized_word'])

tokenized_sentences = pd.Series(train['tokenized_sentence'])

In [ ]:

# Perform topic modeling on both summaries and prompts

print("Performing topic modeling on training text...")

lda_display_summaries = topic_modeling(tokenized_sentences)

print("Performing topic modeling on test text...")

lda_display_prompts = topic_modeling(tokenized_sentences)

# Perform sentiment analysis on summaries and prompts

print("Performing sentiment analysis on train...")

train['sentiment'] = sentiment_analysis(train['corrected_text'])

print("Performing sentiment analysis on test...")

# test['sentiment'] = sentiment_analysis(test['corrected_text'])

# Calculate text complexity for summaries and prompts

print("Calculating text complexity for train...")

train['flesch_score'] = text_complexity(train['corrected_text'])

print("Calculating text complexity for test...")

# test['flesch_score'] = text_complexity(test['corrected_text'])

In [ ]:

# Visualize sentiment and text complexity using histograms
sns.histplot(train['sentiment']).set_title('Sentiment Distribution in Essays')
plt.show()

In [ ]:
print('\n Preparing Visualizations.....')

sns.histplot(train['flesch_score']).set_title('Flesch Reading Ease Score Distribution')
plt.show()


In [ ]:
# Named Entity Recognition (Limiting to first 100 rows for demonstration)
nlp = spacy.load("en_core_web_sm")
texts = ' '.join(train['corrected_text'][:100])
doc_text = nlp(texts)

print('\n Preparing Visualizations.....')

# Count the frequencies of named entity types in summaries and prompts
entity_freq_summaries = Counter([ent.label_ for ent in doc_text.ents])

# Visualize named entity frequencies
plt.figure(figsize=(10, 6))
plt.bar(entity_freq_summaries.keys(), entity_freq_summaries.values())
plt.title('Named Entity Frequency')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Preprocessing
# Placeholder for preprocessing code


# <p style="font-family:JetBrains Mono; font-weight:bold; letter-spacing: 2px; color:#243139; font-size:140%; text-align:left;padding: 0px; border-bottom: 3px solid #000000"> <a name="training"></a>Training</p>

In [ ]:
def custom_train_validation_split(essays, test_size=0.2, random_state=56):
    
    """
    Custom function to perform train-validation split ensuring that
    the same prompt IDs are in both training and validation sets.

    Parameters:
    - summaries: DataFrame containing summaries and associated prompt_ids
    - prompts: DataFrame containing prompts and associated prompt_ids
    - test_size: Proportion of the dataset to be used as the validation set
    - random_state: Random seed for reproducibility

    Returns:
    - train_summaries: Training set containing summaries
    - validation_summaries: Validation set containing summaries
    - train_prompts: Training set containing prompts
    - validation_prompts: Validation set containing prompts
    """
    
    # Extract unique prompt IDs
    unique_essay_ids = essays['essay_id'].unique()

    # Split the unique prompt IDs into training and validation sets
    train_ids, validation_ids = train_test_split(unique_essay_ids, test_size=test_size, random_state=random_state)

    # Use these IDs to filter the original summaries and prompts DataFrames
    train_essays = essays[essays['essay_id'].isin(train_ids)]
    validation_essays = essays[essays['essay_id'].isin(validation_ids)]

    return train_essays, validation_essays


In [ ]:
train_essays, validation_essays = custom_train_validation_split(train, test_size=0.25, random_state=42)

In [ ]:
train_essays.head()

In [ ]:


def extract_features(essay, col = 'corrected_text', tfidf_vectorizer=None):
    
    """
    Extracts TF-IDF features from a list of texts.
    
    Parameters:
    - texts (list): A list of strings containing the text to be processed.
    - tfidf_vectorizer (TfidfVectorizer, optional): A pre-fitted TfidfVectorizer. If None, a new one will be fitted.
    
    Returns:
    - array: The TF-IDF features in dense array format.
    - TfidfVectorizer: The fitted or provided TfidfVectorizer instance.
    """
    
    import pandas as pd
    from sklearn.feature_extraction.text import TfidfVectorizer



    vectorizer = TfidfVectorizer(
            tokenizer=lambda x: x,
            preprocessor=lambda x: x,
            token_pattern=None,
            strip_accents='unicode',
            analyzer = 'word',
            ngram_range=(1,5),
            min_df=0.05,
            max_df=0.95,
            sublinear_tf=True,)

    train_tfid = vectorizer.fit_transform([i for i in essay[col]])
    dense_matrix = train_tfid.toarray()
    df = pd.DataFrame(dense_matrix)
    tfid_columns = [ f'tfid_{i}' for i in range(len(df.columns))]
    df.columns = tfid_columns
    df['essay_id'] = essay['essay_id']
    train_feats = essay.merge(df, on='essay_id', how='left')

    return train_feats, tfidf_vectorizer




def preprocess_data(essays, tfidf_vectorizer=None):
    
    """
    Preprocesses and computes features for a dataset with text summaries and prompts.
    
    Parameters:
    - summaries (DataFrame): DataFrame containing the text summaries.
    - prompts (DataFrame): DataFrame containing the text prompts.
    - tfidf_vectorizer (TfidfVectorizer, optional): A pre-fitted TfidfVectorizer.
    
    Returns:
    - DataFrame: The preprocessed and feature-engineered DataFrame.
    """
    
    print('Preprocessing Data.......')

    # Extract TF-IDF features

    essays, tfidf_vectorizer = extract_features(essays, col='corrected_text',
                                                         tfidf_vectorizer=tfidf_vectorizer)

    stop_words = set(stopwords.words('english'))
    
    print('Generating Text Based Features.......')

    # Compute word count, sentence count, text length, and stopword count
    essays['word_count'] = essays['corrected_text'].apply(lambda x: len(word_tokenize(x)))
    essays['sentence_count'] = essays['corrected_text'].apply(lambda x: len(sent_tokenize(x)))
    essays['len_text'] = essays['corrected_text'].str.len()
    essays['stop_count'] = essays['corrected_text'].apply(lambda x: len([word for word in word_tokenize(x) if word in stop_words]))
    
    import string

    essays['punct_count'] = essays['corrected_text'].apply(lambda x: len([char for char in x if char in string.punctuation]))
    essays['capital_count'] = essays['corrected_text'].apply(lambda x: len([word for word in word_tokenize(x) if word.isupper()]))
    
    from nltk import pos_tag

    essays['noun_count'] = essays['corrected_text'].apply(lambda x: len([word for word, pos in pos_tag(word_tokenize(x)) if pos.startswith('NN')]))
    
    from nltk import ne_chunk

    essays['ne_count'] = essays['corrected_text'].apply(lambda x: len([chunk for chunk in ne_chunk(pos_tag(word_tokenize(x))) if hasattr(chunk, 'label')]))
    essays['avg_word_len'] = essays['corrected_text'].apply(lambda x: sum(len(word) for word in word_tokenize(x)) / len(word_tokenize(x)) if len(word_tokenize(x)) > 0 else 0)
    essays['lex_div'] = essays['corrected_text'].apply(lambda x: len(set(word_tokenize(x))) / len(word_tokenize(x)) if len(word_tokenize(x)) > 0 else 0)
    
    from textblob import TextBlob

    essays['polarity'] = essays['corrected_text'].apply(lambda x: TextBlob(x).sentiment.polarity)
    essays['subjectivity'] = essays['corrected_text'].apply(lambda x: TextBlob(x).sentiment.subjectivity)
    
    from collections import Counter

    essays['most_common_word_count'] = essays['corrected_text'].apply(lambda x: Counter(word_tokenize(x)).most_common(1)[0][1] if len(word_tokenize(x)) > 0 else 0)

    
    # # Calculate cosine similarity between the text and its corresponding prompt
    # print('Computing Cosine Similarity.......')
    # merged_data = compute_cosine_similarity(merged_data, 
    #                                         'treated_question_summaries', 
    #                                         'treated_question_prompts')
    
    return essays, tfidf_vectorizer


def compute_cosine_similarity(df, text_col, content_col):
    
    """
    Computes the cosine similarity between two columns of text in a DataFrame.
    
    Parameters:
    - df (DataFrame): The DataFrame containing the texts.
    - text_col (str): The name of the column containing the first set of texts.
    - content_col (str): The name of the column containing the second set of texts.
    
    Returns:
    - DataFrame: The DataFrame with an additional column for the computed cosine similarity.
    """
    
    # Combine texts from both columns to fit the TF-IDF vectorizer
    all_texts = df[text_col].tolist() + df[content_col].tolist()
    
    # Fit the TF-IDF vectorizer on the combined corpus
    vectorizer = TfidfVectorizer()
    vectorizer.fit(all_texts)
    
    # Generate TF-IDF vectors for both columns
    text_tfidf = vectorizer.transform(df[text_col])
    content_tfidf = vectorizer.transform(df[content_col])
    
    # Compute cosine similarity for each pair of text and content
    cosine_sim_values = [cosine_similarity(text_tfidf[i], content_tfidf[i])[0][0] for i in range(len(df))]
    
    # Add the computed cosine similarity values to the DataFrame
    df['cos_sim'] = cosine_sim_values
    
    return df


In [ ]:
train_essays, tfidf_vectorizer = preprocess_data(train_essays)

In [ ]:
validation_essays, _ = preprocess_data(validation_essays, tfidf_vectorizer=tfidf_vectorizer)

In [ ]:
train_essays['word_tokens'][0]

In [ ]:
train_essays.head()

In [ ]:
validation_essays.head()

In [ ]:
train_essays.columns

In [ ]:
len(train_essays.columns)

In [ ]:
drop_cols = ['sent_text','corrected_text', 'lowered',
       'clean_text', 'no_punct', 'word_tokens', 'sent_tokens',
       'tokenized_word','tokenized_sentence']

training = train_essays.copy()
validation = validation_essays.copy()

training.drop(columns=drop_cols, inplace= True)
validation_essays.drop(columns=drop_cols, inplace= True)



In [ ]:
training.head()

In [ ]:
validation.head()

In [ ]:
# training.to_parquet('data/training_features.parquet')
# validation.to_parquet('data/validation_features.parquet')

In [ ]:
feature_cols = []

for col in training.columns:
    if (col != 'essay_id') and (col != 'score') and (col != 'clean_text'):
        feature_cols.append(col) 


# Select relevant columns (replace with actual column names)

training_labels = training['score']

validation_labels = validation['score']

train_features = training[feature_cols].values

val_features = validation[feature_cols].values

# test_features = test[feature_cols].values


# Standardize the features if needed

scaler = StandardScaler()


train_features = scaler.fit_transform(train_features)

val_features = scaler.transform(val_features)

# test_features = scaler.transform(test_features)



# Optionally, convert back to DataFrame

train_labels = pd.DataFrame(training_labels, columns=['score'],
                             index=training_labels.index)

val_labels = pd.DataFrame(validation_labels, columns=['score'], 
                          index=validation_labels.index)

import pickle


with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
    


In [ ]:
scaler_path = 'scaler.pkl'

# Check if the file has been written correctly and is not empty
import os

if os.path.getsize(scaler_path) > 0:
    print(f"Scaler saved successfully in {scaler_path}.")
else:
    print(f"Failed to save scaler to {scaler_path}. File is empty.")

# <p style="font-family:JetBrains Mono; font-weight:bold; letter-spacing: 2px; color:#243139; font-size:140%; text-align:left;padding: 0px; border-bottom: 3px solid #000000"> <a name="inference"></a>Inference</p>

In [ ]:
# Inference
# Placeholder for inference code


# <p style="font-family:JetBrains Mono; font-weight:bold; letter-spacing: 2px; color:#243139; font-size:140%; text-align:left;padding: 0px; border-bottom: 3px solid #000000"> <a name="model-evaluation"></a>Model Evaluation</p>

In [ ]:
# Model Evaluation
# Placeholder for model evaluation code
